In [2]:
import pandas as pd
import numpy as np

In [6]:
data = pd.read_csv("auto24_vehicles_nov25_wave3.csv")

#Cleaning years - dropping missing values from year, power, fuel_type, transmission, make columns. 
data = data.dropna(subset=["year", "power_ps", "fuel_type", "transmission", "make"])

#Converting year float to int and keeping only years between 2005-2027
data["year"] = data["year"].astype(int)
data = data.loc[(data["year"] < 2027) & (data["year"] > 2005)]

#Cleaning body types - most of the ads about non-passenger cars are classified as other (trucks, bulldozers, motorcycles, boats etc.)
data = data.loc[(data["body_type"] != "other")]

#cleaning up prices - everything between 500 - 100 000 EUR is kept.
data = data.loc[(data["price_eur"] > 500) & (data["price_eur"] < 100000)]

data["make"] = (data["make"].str.lower().str.strip()
    .str.replace("-", " ", regex=False)
    .replace({"mercedes amg": "mercedes benz"}))

data["model"] = (
    data["model"].astype(str).str.lower().str.strip()
    .str.replace(r"[-_/]+", " ", regex=True)
    .str.replace(r"\b([a-z])\s+(\d+)\b", r"\1\2", regex=True)
)

#aligning the fuel types, transmission and body types between the ebay and auto24 datasets
data["fuel_type"] = data["fuel_type"].replace({"plug_in_hybrid": "hybrid", "hydrogen": "other"})
data["transmission"] = data["transmission"].replace({"semi_automatic": "automatic"})
data["body_type"] = data["body_type"].replace({"van": "minivan", "pickup": "suv"})

data["car_age"] = 2026 - data["year"]

data.to_csv("auto24_topredict_ebay.csv")